In [1]:
import pandas as pd

# 1. Lecture des fichiers (adaptez les noms des fichiers selon votre ordinateur)
# On utilise sep='\t' car vos données semblent séparées par des tabulations (TSV)
df_sales = pd.read_csv('Details_Maroc.csv')
df_customers = pd.read_csv('Orders_Maroc.csv')
df_sales

,Order ID,Total Sales,Profit,Quantity,Category,Sub-Category,PaymentMode
0,M-10000,1096,658,7,Electronics,Electronic Games,Paiement à la livraison
1,M-10001,5729,64,14,Furniture,Chairs,Paiement en plusieurs fois
2,M-10002,2927,146,8,Furniture,Bookcases,Paiement en plusieurs fois
3,M-10003,2847,712,8,Electronics,Printers,Carte bancaire
4,M-10004,2617,1151,4,Electronics,Phones,Carte bancaire
...,...,...,...,...,...,...,...
1495,M-11495,7,-3,2,Clothing,Hankerchief,Paiement à la livraison
1496,M-11496,3151,-35,7,Clothing,Trousers,Paiement en plusieurs fois
1497,M-11497,4141,1698,13,Electronics,Printers,Paiement à la livraison
1498,M-11498,7,-2,1,Clothing,Hankerchief,Paiement à la livraison


In [2]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# ==========================================
# 1. CHARGEMENT DE VOS DONNÉES RÉELLES
# ==========================================
# Remplacez les noms de fichiers par les vôtres
try:
    df_sales = pd.read_csv("Sales_Data.csv") # ou pd.read_excel
    df_cust = pd.read_csv("Customer_Data.csv")
except:
    # Si les fichiers ne sont pas là, ce bloc crée un échantillon basé sur ce que vous m'avez montré
    print("Fichiers non trouvés, utilisation de l'échantillon réel fourni...")
    data_sales = {
        'Order ID': ['M-10000', 'M-10001', 'M-10002', 'M-10006', 'M-11492'],
        'Total Sales': [1096, 5729, 2927, 275, 2830],
        'Profit': [658, 64, 146, -275, -1981],
        'Quantity': [7, 14, 8, 4, 13],
        'Category': ['Electronics', 'Furniture', 'Furniture', 'Clothing', 'Furniture'],
        'Sub-Category': ['Electronic Games', 'Chairs', 'Bookcases', 'Saree', 'Bookcases'],
        'PaymentMode': ['Paiement à la livraison', 'Paiement en plusieurs fois', 'Paiement à la livraison', 'Virement bancaire', 'Paiement en plusieurs fois']
    }
    data_cust = {
        'Order ID': ['M-10000', 'M-10001', 'M-10002'],
        'Order Date': ['10/3/2018', '3/2/2018', '24/1/2018'],
        'CustomerName': ['Sara', 'Nadia', 'Mohamed'],
        'State': ['Rabat-Salé-Kénitra', 'Marrakech-Safi', 'Casablanca-Settat'],
        'City': ['Kénitra', 'Marrakech', 'Mohammedia']
    }
    df_sales = pd.DataFrame(data_sales)
    df_cust = pd.DataFrame(data_cust)

# ==========================================
# 2. FUSION ET RÉPARATION DES DONNÉES
# ==========================================
# Jointure (Left Join) pour garder toutes les ventes
df_merged = pd.merge(df_sales, df_cust, on="Order ID", how="left")

# Réparation des 1000 lignes manquantes (State, City, Date) 
# On utilise les probabilités des données existantes pour remplir le vide
for col in ['State', 'City', 'Order Date', 'CustomerName']:
    valid_data = df_merged[col].dropna().values
    df_merged[col] = df_merged[col].apply(lambda x: random.choice(valid_data) if pd.isnull(x) else x)

# ==========================================
# 3. EXTRAPOLATION À 80 000 LIGNES
# ==========================================
print("Génération de 80 000 lignes...")
df_80k = df_merged.sample(n=80000, replace=True).reset_index(drop=True)

# Mise à jour des Order ID pour qu'ils soient uniques
df_80k['Order ID'] = [f"M-{10000 + i}" for i in range(80000)]

# ==========================================
# 4. INJECTION DE CRÉATIVITÉ (SECTIONS PFE)
# ==========================================
# A. Dates modernes (2023-2024)
start_date = datetime(2023, 1, 1)
df_80k['Order Date'] = [start_date + timedelta(days=random.randint(0, 500)) for _ in range(80000)]

# B. Profiling Client
df_80k['Gender'] = np.random.choice(['Homme', 'Femme'], size=80000, p=[0.48, 0.52])
df_80k['Age_Group'] = np.random.choice(['18-25', '26-35', '36-50', '50+'], size=80000)

# C. Analyse Marketing (Canaux)
df_80k['Marketing_Channel'] = np.random.choice(['Instagram Ads', 'Facebook Ads', 'Google Search', 'SEO Naturel', 'Influenceur'], size=80000)

# D. Analyse Logistique (Statuts de livraison basés sur le Profit)
# On simule que si le profit est très négatif, c'est souvent un retour (frais logistiques)
def assign_status(profit):
    if profit < -200: return 'Retourné'
    if profit < 0: return 'Annulé'
    return 'Livré'

df_80k['Delivery_Status'] = df_80k['Profit'].apply(assign_status)

# E. Ajout du Coût d'achat (pour analyse de marge)
# Coût = Vente - Profit
df_80k['Cost_Price'] = df_80k['Total Sales'] - df_80k['Profit']

# F. Ajout de "Bruit" (jitter) pour que les prix ne soient pas identiques
df_80k['Total Sales'] = (df_80k['Total Sales'] * np.random.uniform(0.9, 1.1, 80000)).round(2)
df_80k['Profit'] = (df_80k['Profit'] * np.random.uniform(0.9, 1.1, 80000)).round(2)

# ==========================================
# 5. NORMALISATION (EXPORT DES 4 TABLES)
# ==========================================
# Table de Faits : Ventes
fact_sales = df_80k[['Order ID', 'Order Date', 'Total Sales', 'Profit', 'Cost_Price', 'Quantity', 'PaymentMode', 'Delivery_Status', 'Marketing_Channel']]

# Table Dimension : Clients
dim_customers = df_80k[['Order ID', 'CustomerName', 'Gender', 'Age_Group']].drop_duplicates()

# Table Dimension : Géographie
dim_geo = df_80k[['Order ID', 'State', 'City']].drop_duplicates()

# Table Dimension : Produits
dim_products = df_80k[['Order ID', 'Category', 'Sub-Category']].drop_duplicates()

# EXPORT FINAL
fact_sales.to_csv("Fact_Sales_PFE.csv", index=False, encoding='utf-8-sig')
dim_customers.to_csv("Dim_Customers_PFE.csv", index=False, encoding='utf-8-sig')
dim_geo.to_csv("Dim_Geography_PFE.csv", index=False, encoding='utf-8-sig')
dim_products.to_csv("Dim_Products_PFE.csv", index=False, encoding='utf-8-sig')

print("Terminé ! Les 4 tables 'PFE_Expert' sont prêtes dans votre dossier.")

Fichiers non trouvés, utilisation de l'échantillon réel fourni...
Génération de 80 000 lignes...
Terminé ! Les 4 tables 'PFE_Expert' sont prêtes dans votre dossier.


In [3]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# --- ÉTAPE 1 : RÉFÉRENTIEL DES NOMS MAROCAINS ---
# On définit qui est Homme et qui est Femme pour garder la cohérence
noms_femmes = ["Sara", "Nadia", "Fatima", "Khadija", "Zineb", "Salma", "Amina", "Meriem", "Layla", "Latifa"]
noms_hommes = ["Mohamed", "Omar", "Ahmed", "Rachid", "Mustapha", "Youssef", "Karim", "Driss", "Hassan", "Hamza"]

# --- ÉTAPE 2 : CHARGEMENT / PRÉPARATION (Comme avant) ---
# ... (votre chargement de fichier sales_raw et cust_raw) ...
# On s'assure que les noms dans votre data d'origine sont bien classés
tous_les_noms = noms_hommes + noms_femmes

# --- ÉTAPE 3 : GÉNÉRATION DES 80 000 LIGNES ---
df_80k = df_merged.sample(n=80000, replace=True).reset_index(drop=True)
df_80k['Order ID'] = [f"M-{10000 + i}" for i in range(80000)]

# --- ÉTAPE 4 : ATTRIBUTION COHÉRENTE DU GENRE ---
# Au lieu du random complet, on crée une fonction de mapping
def attribuer_genre(nom):
    if nom in noms_femmes:
        return "Femme"
    if nom in noms_hommes:
        return "Homme"
    return random.choice(["Homme", "Femme"]) # Au cas où un nom inconnu apparaît

# On force d'abord l'attribution de noms variés (pour éviter d'avoir que 3 noms)
df_80k['CustomerName'] = [random.choice(tous_les_noms) for _ in range(80000)]

# On applique le genre en fonction du nom
df_80k['Gender'] = df_80k['CustomerName'].apply(attribuer_genre)

# --- ÉTAPE 5 : AJOUT DES AUTRES COLONNES (SECTION PFE) ---
start_date = datetime(2023, 1, 1)
df_80k['Order Date'] = [start_date + timedelta(days=random.randint(0, 500)) for _ in range(80000)]
df_80k['Age_Group'] = np.random.choice(['18-25', '26-35', '36-50', '50+'], size=80000)
df_80k['Marketing_Channel'] = np.random.choice(['Instagram Ads', 'Facebook Ads', 'Google Search', 'SEO Naturel'], size=80000)
df_80k['Delivery_Status'] = df_80k['Profit'].apply(lambda x: 'Retourné' if x < -200 else ('Annulé' if x < 0 else 'Livré'))
df_80k['Cost_Price'] = (df_80k['Total Sales'] - df_80k['Profit']).round(2)

# --- ÉTAPE 6 : EXPORT DES TABLES NORMALISÉES ---
fact_sales = df_80k[['Order ID', 'Order Date', 'Total Sales', 'Profit', 'Cost_Price', 'Quantity', 'PaymentMode', 'Delivery_Status', 'Marketing_Channel']]
dim_customers = df_80k[['Order ID', 'CustomerName', 'Gender', 'Age_Group', 'State', 'City']].drop_duplicates()
dim_products = df_80k[['Order ID', 'Category', 'Sub-Category']].drop_duplicates()

# Export final sans erreur de genre
fact_sales.to_csv("Fact_Sales_PFE.csv", index=False, encoding='utf-8-sig')
dim_customers.to_csv("Dim_Customers_PFE.csv", index=False, encoding='utf-8-sig')
dim_products.to_csv("Dim_Products_PFE.csv", index=False, encoding='utf-8-sig')

print("Fichiers exportés avec succès ! La cohérence Nom/Genre est respectée.")

Fichiers exportés avec succès ! La cohérence Nom/Genre est respectée.


In [4]:
# --- AJOUT DES VARIABLES STRATÉGIQUES ---

# 1. On définit des coûts d'acquisition par canal (en DH)
cac_mapping = {
    'Instagram Ads': 45, 
    'Facebook Ads': 35, 
    'Google Search': 25, 
    'SEO Naturel': 5, 
    'Influenceur': 60
}

# 2. On enrichit la génération
data_expert = []
for i in range(80000):
    # (Logique précédente de base...)
    sales = df_80k.iloc[i]['Total Sales']
    profit_brut = df_80k.iloc[i]['Profit']
    channel = df_80k.iloc[i]['Marketing_Channel']
    
    # --- NOUVELLES COLONNES ---
    # Coût marketing
    marketing_cost = cac_mapping[channel] + random.randint(-5, 5)
    
    # Performance Logistique
    promised_days = random.choice([2, 3, 4])
    actual_days = promised_days + random.choice([0, 0, 0, 1, 2, 5]) # Parfois du retard
    
    # Satisfaction Client (Rating)
    # Si retard -> la note baisse
    if actual_days > promised_days:
        rating = random.randint(1, 3)
    else:
        rating = random.randint(3, 5)
        
    # Calcul du Profit Net (Marge après Marketing et Logistique)
    shipping_fee = 35 # Prix moyen au Maroc
    profit_net = profit_brut - marketing_cost - (shipping_fee if actual_days > 5 else 0)

    data_expert.append([marketing_cost, promised_days, actual_days, rating, profit_net])

# On ajoute ces colonnes au DataFrame final
# (C'est ici que votre PFE devient puissant)

In [8]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# ==========================================
# 1. RÉFÉRENTIELS ET PARAMÈTRES (ADN MAROC)
# ==========================================
n_rows = 80000

noms_femmes = ["Sara", "Nadia", "Fatima", "Khadija", "Zineb", "Salma", "Amina", "Meriem", "Layla", "Latifa"]
noms_hommes = ["Mohamed", "Omar", "Ahmed", "Rachid", "Mustapha", "Youssef", "Karim", "Driss", "Hassan", "Hamza"]

# Entreprises de livraison et leurs caractéristiques (Taux de succès)
delivery_partners = {
    "Aramex": 0.75,         # Grand volume, taux moyen
    "Speedaf": 0.78,        # Très présent e-commerce
    "CTM Messagerie": 0.82, # Fiable pour gros colis
    "Ghazala": 0.80,        # Spécialiste mobilier
    "Livreurs Privés": 0.92 # Très efficace en centre-ville
}

marketing_channels = {
    "Instagram Ads": 45, "Facebook Ads": 35, "Google Search": 25, "SEO Naturel": 5, "Influenceur": 60
}

# ==========================================
# 2. CHARGEMENT / SIMULATION DATA RÉELLE
# ==========================================
# Le code utilise vos données réelles comme base statistique
data_sales = {
    'Order ID': ['M-10000', 'M-10001', 'M-10002', 'M-10006', 'M-11492'],
    'Total Sales': [1096, 5729, 2927, 275, 2830],
    'Profit': [658, 64, 146, -275, -1981],
    'Quantity': [7, 14, 8, 4, 13],
    'Category': ['Electronics', 'Furniture', 'Furniture', 'Clothing', 'Furniture'],
    'Sub-Category': ['Electronic Games', 'Chairs', 'Bookcases', 'Saree', 'Bookcases'],
    'PaymentMode': ['Paiement à la livraison', 'Paiement en plusieurs fois', 'Paiement à la livraison', 'Virement bancaire', 'Paiement en plusieurs fois']
}
data_cust = {
    'Order ID': ['M-10000', 'M-10001', 'M-10002'],
    'Order Date': ['10/3/2018', '3/2/2018', '24/1/2018'],
    'CustomerName': ['Sara', 'Nadia', 'Mohamed'],
    'State': ['Rabat-Salé-Kénitra', 'Marrakech-Safi', 'Casablanca-Settat'],
    'City': ['Kénitra', 'Marrakech', 'Mohammedia']
}

df_sales_base = pd.DataFrame(data_sales)
df_cust_base = pd.DataFrame(data_cust)

# Fusion et réparation (remplissage des vides avec des données cohérentes)
df_merged = pd.merge(df_sales_base, df_cust_base, on="Order ID", how="left")
for col in ['State', 'City', 'Order Date']:
    valid_data = df_cust_base[col].dropna().values
    df_merged[col] = df_merged[col].apply(lambda x: random.choice(valid_data) if pd.isnull(x) else x)

# ==========================================
# 3. GÉNÉRATION DES 80 000 LIGNES
# ==========================================
df_80k = df_merged.sample(n=n_rows, replace=True).reset_index(drop=True)
df_80k['Order ID'] = [f"M-{10000 + i}" for i in range(n_rows)]

# --- Attribution Nom et Genre Cohérent ---
tous_noms = noms_hommes + noms_femmes
df_80k['CustomerName'] = [random.choice(tous_noms) for _ in range(n_rows)]
df_80k['Gender'] = df_80k['CustomerName'].apply(lambda x: "Femme" if x in noms_femmes else "Homme")

# --- Attribution Marketing et Acquisition ---
df_80k['Marketing_Channel'] = [random.choice(list(marketing_channels.keys())) for _ in range(n_rows)]
df_80k['CAC'] = df_80k['Marketing_Channel'].map(marketing_channels)

# --- Attribution Entreprise de Livraison ---
def assign_logistic(cat, city):
    if cat == "Furniture": return random.choice(["CTM Messagerie", "Ghazala"])
    if city in ["Casablanca", "Rabat", "Marrakech"]: return random.choice(["Livreurs Privés", "Speedaf", "Aramex"])
    return random.choice(["Aramex", "Speedaf", "CTM Messagerie"])

df_80k['Delivery_Partner'] = df_80k.apply(lambda x: assign_logistic(x['Category'], x['City']), axis=1)

# --- Statut de Livraison (Logique de performance) ---
def determine_status(partner):
    success_rate = delivery_partners[partner]
    rand = random.random()
    if rand < success_rate: return "Livré"
    if rand < success_rate + 0.10: return "Annulé" # 10% annulation avant envoi
    return "Retourné" # Le reste est retourné après tentative

df_80k['Delivery_Status'] = df_80k['Delivery_Partner'].apply(determine_status)

# --- Calculs Financiers (Marge Nette) ---
df_80k['Shipping_Fee'] = 35 # Prix moyen fixe par envoi
# Marge Nette = Profit Brut - Coût Marketing - Frais port (si retourné, le port est perdu)
df_80k['Net_Profit'] = df_80k['Profit'] - df_80k['CAC'] - df_80k['Shipping_Fee']

# Dates modernes
start_date = datetime(2023, 1, 1)
df_80k['Order Date'] = [start_date + timedelta(days=random.randint(0, 500)) for _ in range(n_rows)]

# ==========================================
# 4. NORMALISATION ET EXPORT (SCHÉMA EN ÉTOILE)
# ==========================================

# 1. Fact_Sales
fact_sales = df_80k[['Order ID', 'Order Date', 'Total Sales', 'Profit', 'Net_Profit', 'Quantity', 
                     'PaymentMode', 'Delivery_Status', 'Delivery_Partner', 'Marketing_Channel']]

# 2. Dim_Customers
dim_customers = df_80k[['Order ID', 'CustomerName', 'Gender', 'State', 'City']].drop_duplicates()

# 3. Dim_Products
dim_products = df_80k[['Order ID', 'Category', 'Sub-Category']].drop_duplicates()

# EXPORTATION
fact_sales.to_csv("Fact_Sales_Final.csv", index=False, encoding='utf-8-sig')
dim_customers.to_csv("Dim_Customers_Final.csv", index=False, encoding='utf-8-sig')
dim_products.to_csv("Dim_Products_Final.csv", index=False, encoding='utf-8-sig')

print("Félicitations ! Votre dataset PFE expert est généré avec 80 000 lignes.")
print("Fichiers créés : Fact_Sales_Final.csv, Dim_Customers_Final.csv, Dim_Products_Final.csv")

Félicitations ! Votre dataset PFE expert est généré avec 80 000 lignes.
Fichiers créés : Fact_Sales_Final.csv, Dim_Customers_Final.csv, Dim_Products_Final.csv


In [9]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# ==========================================
# 1. PARAMÈTRES ET RÉFÉRENTIELS RÉELS
# ==========================================
n_rows = 80000

# --- NOMS (F: 50+, M: 60+) ---
femmes = ["Sara", "Nadia", "Fatima", "Khadija", "Zineb", "Salma", "Amina", "Meriem", "Layla", "Latifa", "Houda", "Imane", "Sanae", "Bouchra", "Raja", "Ghita", "Kenza", "Soukaina", "Meryem", "Asmae", "Hajar", "Wiam", "Ikram", "Nawal", "Fatiha", "Majda", "Hasna", "Malika", "Siham", "Randa", "Loubna", "Keltoum", "Oumaima", "Chaimae", "Yasmine", "Dounia", "Hiba", "Nisrine", "Lamia", "Zahra", "Souad", "Ranya", "Intissar", "Basma", "Safae", "Kaoutar", "Nouzha", "Bahia", "Rabia", "Touria"]
hommes = ["Mohamed", "Omar", "Ahmed", "Rachid", "Mustapha", "Youssef", "Karim", "Driss", "Hassan", "Hamza", "Adil", "Anas", "Yassine", "Simohamed", "Amine", "Mehdi", "Abdellah", "Saad", "Tarik", "Brahim", "Khalid", "Jawad", "Ismail", "Zakaria", "Othmane", "Walid", "Fouad", "Aziz", "Reda", "Nabil", "Ali", "Kamal", "Said", "Jalal", "Hicham", "Mounir", "Abdelaziz", "Noureddine", "Soufiane", "Ilyas", "Ayoub", "Faycal", "Badr", "Marouane", "Abderrahmane", "Aissam", "Oussama", "Abdelhak", "Jaouad", "Salah", "Taha", "Younes", "Rayan", "Faissal", "Redouane", "Issam", "Zouhair", "Ghali", "Habib", "Nouamane"]

# --- CATALOGUE PRODUITS (50+) ---
produits = {
    "Electronics": ["iPhone 15", "Samsung S23", "MacBook Air", "Ecouteurs Bluetooth", "PowerBank 20k", "Smart Watch Xiaomi", "Imprimante HP", "PlayStation 5", "PC Gamer Asus", "Clé USB 64GB", "Tablette Lenovo", "Appareil Photo Canon", "Enceinte JBL", "TV LG 55", "Sèche-cheveux Dyson"],
    "Clothing": ["Djellaba Moderne", "T-shirt Oversize", "Jean Slim Fit", "Basket Nike", "Abaya Soie", "Caftan de fête", "Chemise en lin", "Sweat à capuche", "Sandales Cuir", "Robe d'été", "Veste Denim", "Costume Homme", "Pyjama Satin", "Echarpe Cachemire", "Ceinture Cuir"],
    "Furniture": ["Tapis Berbère", "Lampe de chevet", "Table basse bois", "Canapé 3 places", "Etagère Murale", "Miroir Doré", "Chaise de bureau", "Bureau d'angle", "Rideaux Occultants", "Cadre déco", "Lit Double", "Armoire 2 portes", "Pouf Salon", "Console Entrée", "Vase Céramique"],
    "Accessories": ["Montre Luxe", "Sac à main cuir", "Lunettes de soleil", "Portefeuille", "Bijoux Argent", "Casquette", "Parapluie", "Valise Voyage", "Coque iPhone", "Porte-clés"]
}

# --- MARKETING & LOGISTIQUE ---
channels = {"Meta (FB/IG)": 35, "TikTok Ads": 22, "Google Search": 45, "Direct/SEO": 5}
logistique = {"Speedaf": 0.78, "Aramex": 0.75, "CTM Messagerie": 0.82, "Ghazala": 0.80, "Catoni": 0.85}
regions_villes = {
    "Casablanca-Settat": ["Casablanca", "Mohammedia", "Berrechid", "El Jadida"],
    "Rabat-Salé-Kénitra": ["Rabat", "Salé", "Kénitra", "Témara"],
    "Marrakech-Safi": ["Marrakech", "Safi", "Essaouira"],
    "Fès-Meknès": ["Fès", "Meknès", "Taza"],
    "Tanger-Tétouan": ["Tanger", "Tétouan", "Al Hoceima"],
    "Souss-Massa": ["Agadir", "Inezgane"]
}

# ==========================================
# 2. GÉNÉRATION DE LA DATA
# ==========================================
data = []
start_date = datetime(2023, 1, 1)

for i in range(n_rows):
    # Identité & Genre
    if random.random() > 0.45:
        name, gender = random.choice(femmes), "Femme"
    else:
        name, gender = random.choice(hommes), "Homme"
    
    age = random.randint(20, 68)
    
    # Date & Saisonnalité
    date = start_date + timedelta(days=random.randint(0, 450))
    month = date.month
    
    # Boost volume selon saison marocaine
    weight = 1
    if month == 3: weight = 1.3 # Ramadan
    if month == 6: weight = 1.4 # Aïd
    if month == 9: weight = 1.2 # Rentrée
    if month == 11: weight = 1.6 # Black Friday

    # Localisation
    region = random.choice(list(regions_villes.keys()))
    city = random.choice(regions_villes[region])
    
    # Produit & Prix
    cat = random.choice(list(produits.keys()))
    prod = random.choice(produits[cat])
    qty = random.randint(1, 4)
    
    if cat == "Electronics": price = random.randint(300, 12000)
    elif cat == "Furniture": price = random.randint(200, 5000)
    else: price = random.randint(50, 1500)
    
    total_sales = price * qty
    cost_price = total_sales * random.uniform(0.5, 0.7)
    
    # Marketing
    channel = random.choice(list(channels.keys()))
    cac = channels[channel] * random.uniform(0.8, 1.2)
    
    # Logistique & Paiement
    pay_mode = random.choice(["Paiement à la livraison", "Paiement à la livraison", "Carte bancaire", "Virement"])
    partner = random.choice(list(logistique.keys()))
    
    # Taux de succès impacté par le mode de paiement (COD est plus risqué)
    success_rate = logistique[partner]
    if pay_mode == "Paiement à la livraison":
        success_rate -= 0.15
        
    rand_success = random.random()
    if rand_success < success_rate:
        status = "Livré"
        profit = total_sales - cost_price - cac
    elif rand_success < success_rate + 0.10:
        status = "Annulé"
        profit = -cac
    else:
        status = "Retourné"
        profit = -(cac + 35) # Perte transport retour (35 DH)

    data.append([
        f"M-{10000+i}", date.strftime("%Y-%m-%d"), name, gender, age, 
        region, city, cat, prod, qty, round(total_sales, 2), 
        round(profit, 2), pay_mode, channel, round(cac, 2), 
        partner, status
    ])

# ==========================================
# 3. CRÉATION DU DATAFRAME ET EXPORT
# ==========================================
columns = [
    "ID_Commande", "Date", "Nom_Client", "Genre", "Age", 
    "Region", "Ville", "Categorie", "Produit", "Quantite", "Ventes_Totales", 
    "Profit_Net", "Mode_Paiement", "Canal_Marketing", "Cout_Acquisition", 
    "Transporteur", "Statut_Livraison"
]

df_pfe = pd.DataFrame(data, columns=columns)

# Exportation en un seul fichier CSV propre
df_pfe.to_csv("Dataset_Marketing_Maroc_80K.csv", index=False, encoding='utf-8-sig')

print("Fichier 'Dataset_Marketing_Maroc_80K.csv' généré avec succès !")

Fichier 'Dataset_Marketing_Maroc_80K.csv' généré avec succès !


In [11]:
# --- IMPORTATION DES DONNÉES SCRAPÉES (Exemple) ---
# On crée un dictionnaire basé sur ton fichier Google Maps
transporteurs_reels = {
    "Speedaf Express": {"note": 2.0, "type": "National"},
    "CTM Messagerie": {"note": 4.0, "type": "National"},
    "Livreur Marrakech": {"note": 4.8, "type": "Local"},
    "Colis Privé Maroc": {"note": 4.1, "type": "National"},
    "SousSwift Livraison": {"note": 5.0, "type": "Regional"},
    "Agence ABB Jumia": {"note": 3.5, "type": "Relais"}
}

# --- LOGIQUE DE SIMULATION ---
def calcul_statut_avec_scraping(nom_transporteur):
    score = transporteurs_reels[nom_transporteur]["note"]
    
    # Plus le score est haut, plus on a de chances d'être "Livré"
    # Un score de 5/5 donne ~90% de réussite
    # Un score de 2/5 donne ~65% de réussite
    seuil_reussite = 0.5 + (score / 10) 
    
    rand = random.random()
    if rand < seuil_reussite:
        return "Livré"
    return "Retourné"

# Application à tes 80 000 lignes
df_pfe['Transporteur'] = [random.choice(list(transporteurs_reels.keys())) for _ in range(80000)]
df_pfe['Statut_Livraison'] = df_pfe['Transporteur'].apply(calcul_statut_avec_scraping)

In [14]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# ==========================================
# 1. RÉFÉRENTIELS LOGISTIQUE & PRIX
# ==========================================
n_rows = 80000

# Tarifs de livraison moyens au Maroc (COD inclus)
# Format : {Ville_Destination: Prix_DH}
shipping_rates = {
    "Casablanca": 35, 
    "Mohammedia": 35,
    "Rabat": 40, 
    "Salé": 40, 
    "Témara": 40,
    "Kénitra": 45,
    "Marrakech": 45, 
    "Fès": 50, 
    "Meknès": 50,
    "Tanger": 50,
    "Agadir": 55,
    "Oujda": 60,
    "Laâyoune": 70
}
default_rate = 55 # Pour les autres villes

# Transporteurs scrapés (Nom: Note Google Maps)
logistique_scraped = {
    "Speedaf Express": 2.1,
    "Aramex": 2.5,
    "CTM Messagerie": 4.2,
    "Livreur Privé (Local)": 4.8,
    "Ghazala": 3.9,
    "Catoni": 4.1
}

# ==========================================
# 2. GÉNÉRATION DES 80 000 LIGNES
# ==========================================
data = []
names_f = ["Sara", "Nadia", "Fatima", "Khadija", "Zineb", "Salma", "Amina", "Meriem", "Ghita", "Imane"]
names_m = ["Mohamed", "Omar", "Ahmed", "Rachid", "Mustapha", "Youssef", "Karim", "Hamza", "Mehdi", "Yassine"]

for i in range(n_rows):
    # Identité
    if random.random() > 0.5:
        name, gender = random.choice(names_f), "Femme"
    else:
        name, gender = random.choice(names_m), "Homme"
    
    # Géographie
    city = random.choice(list(shipping_rates.keys()))
    
    # Produit & Catégorie
    cat = random.choice(["Electronics", "Clothing", "Furniture"])
    qty = random.randint(1, 3)
    
    # Calcul des Ventes et Coût d'achat
    if cat == "Electronics": sales = random.randint(500, 8000)
    elif cat == "Furniture": sales = random.randint(300, 4000)
    else: sales = random.randint(50, 800)
    
    cost_price = sales * 0.6 # 60% de coût produit
    
    # --- LOGIQUE LOGISTIQUE (LE COEUR DU PFE) ---
    # 1. Calcul du prix de livraison selon la ville
    delivery_price = shipping_rates.get(city, default_rate)
    
    # Supplément pour mobilier (poids/encombrement)
    if cat == "Furniture":
        delivery_price += 40 
        
    # 2. Choix du transporteur et probabilité de succès (basée sur la note scrapée)
    partner = random.choice(list(logistique_scraped.keys()))
    score = logistique_scraped[partner]
    
    # Calcul du statut (plus la note est haute, plus on livre)
    success_chance = 0.5 + (score / 10)
    if random.random() < success_chance:
        status = "Livré"
        # Marge Nette = CA - Coût - Livraison - Marketing (estimé à 30DH)
        net_profit = sales - cost_price - delivery_price - 30
    else:
        status = "Retourné"
        # Si retourné, on perd le prix de livraison et le marketing
        net_profit = -(delivery_price + 30)

    data.append([
        f"M-{10000+i}", name, gender, city, cat, qty, 
        round(sales, 2), status, partner, delivery_price, round(net_profit, 2)
    ])

# Création du DataFrame
df_final = pd.DataFrame(data, columns=[
    "ID_Order", "Client", "Genre", "Ville", "Categorie", "Quantite", 
    "CA_Brut", "Statut", "Transporteur", "Frais_Livraison", "Marge_Nette"
])

df_final.to_csv("PFE_Marketing_Logistique_Maroc.csv", index=False, encoding='utf-8-sig')
print("Dataset généré avec succès !")
# --- 1. MOTIFS DE RETOUR (Réalité du terrain marocain) ---
motifs_retour = ["Client injoignable", "Refus à la livraison", "Délai trop long", "Produit endommagé", "Changement d'avis"]

# --- 2. TUNNEL DE CONVERSION (Funnel) ---
df_final['Impressions'] = np.random.randint(1000, 5000, size=80000)
df_final['Clics'] = (df_final['Impressions'] * np.random.uniform(0.01, 0.05, size=80000)).astype(int)

# --- 3. FIDÉLISATION ---
df_final['Type_Client'] = np.random.choice(["Nouveau", "Fidèle"], size=80000, p=[0.7, 0.3])

# --- 4. LOGIQUE DES RETOURS ---
def enrichir_retour(row):
    if row['Statut'] == "Retourné":
        # Si le transporteur est mal noté, le motif est souvent "Délai trop long"
        if row['Transporteur'] in ["Speedaf Express", "Aramex"]:
            return random.choice(["Délai trop long", "Client injoignable", "Client injoignable"])
        return random.choice(motifs_retour)
    return "N/A"

df_final['Motif_Retour'] = df_final.apply(enrichir_retour, axis=1)

# --- 5. INDICE HCP (Simplifié) ---
richesse_hcp = {"Casablanca": 95, "Rabat": 90, "Marrakech": 75, "Tanger": 70, "Oujda": 45, "Settat": 40}
df_final['Indice_Richesse_Ville'] = df_final['Ville'].map(richesse_hcp).fillna(50)

PermissionError: [Errno 13] Permission denied: 'PFE_Marketing_Logistique_Maroc.csv'

In [13]:
# --- 1. MOTIFS DE RETOUR (Réalité du terrain marocain) ---
motifs_retour = ["Client injoignable", "Refus à la livraison", "Délai trop long", "Produit endommagé", "Changement d'avis"]

# --- 2. TUNNEL DE CONVERSION (Funnel) ---
df_final['Impressions'] = np.random.randint(1000, 5000, size=80000)
df_final['Clics'] = (df_final['Impressions'] * np.random.uniform(0.01, 0.05, size=80000)).astype(int)

# --- 3. FIDÉLISATION ---
df_final['Type_Client'] = np.random.choice(["Nouveau", "Fidèle"], size=80000, p=[0.7, 0.3])

# --- 4. LOGIQUE DES RETOURS ---
def enrichir_retour(row):
    if row['Statut'] == "Retourné":
        # Si le transporteur est mal noté, le motif est souvent "Délai trop long"
        if row['Transporteur'] in ["Speedaf Express", "Aramex"]:
            return random.choice(["Délai trop long", "Client injoignable", "Client injoignable"])
        return random.choice(motifs_retour)
    return "N/A"

df_final['Motif_Retour'] = df_final.apply(enrichir_retour, axis=1)

# --- 5. INDICE HCP (Simplifié) ---
richesse_hcp = {"Casablanca": 95, "Rabat": 90, "Marrakech": 75, "Tanger": 70, "Oujda": 45, "Settat": 40}
df_final['Indice_Richesse_Ville'] = df_final['Ville'].map(richesse_hcp).fillna(50)

In [3]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# ==========================================
# 1. RÉFÉRENTIELS EXPERTS (MAROC)
# ==========================================
n_rows = 80000

# --- NOMS (F: 50+, M: 60+) ---
femmes = ["Sara", "Nadia", "Fatima", "Khadija", "Zineb", "Salma", "Amina", "Meriem", "Layla", "Latifa", "Houda", "Imane", "Sanae", "Bouchra", "Raja", "Ghita", "Kenza", "Soukaina", "Meryem", "Asmae", "Hajar", "Wiam", "Ikram", "Nawal", "Fatiha", "Majda", "Hasna", "Malika", "Siham", "Randa", "Loubna", "Keltoum", "Oumaima", "Chaimae", "Yasmine", "Dounia", "Hiba", "Nisrine", "Lamia", "Zahra", "Souad", "Ranya", "Intissar", "Basma", "Safae", "Kaoutar", "Nouzha", "Bahia", "Rabia", "Touria", "Malak", "Kenaza"]
hommes = ["Mohamed", "Omar", "Ahmed", "Rachid", "Mustapha", "Youssef", "Karim", "Driss", "Hassan", "Hamza", "Adil", "Anas", "Yassine", "Simohamed", "Amine", "Mehdi", "Abdellah", "Saad", "Tarik", "Brahim", "Khalid", "Jawad", "Ismail", "Zakaria", "Othmane", "Walid", "Fouad", "Aziz", "Reda", "Nabil", "Ali", "Kamal", "Said", "Jalal", "Hicham", "Mounir", "Abdelaziz", "Noureddine", "Soufiane", "Ilyas", "Ayoub", "Faycal", "Badr", "Marouane", "Abderrahmane", "Aissam", "Oussama", "Abdelhak", "Jaouad", "Salah", "Taha", "Younes", "Rayan", "Faissal", "Redouane", "Issam", "Zouhair", "Ghali", "Habib", "Nouamane", "Anwar", "Brahim", "Sami"]

# --- PRODUITS (50+) ---
produits = {
    "Electronics": ["iPhone 15", "Samsung S23", "MacBook Air", "Ecouteurs Bluetooth", "PowerBank 20k", "Smart Watch Xiaomi", "Imprimante HP", "PlayStation 5", "PC Gamer Asus", "Clé USB 64GB", "Tablette Lenovo", "Appareil Photo Canon", "Enceinte JBL", "TV LG 55", "Sèche-cheveux Dyson"],
    "Clothing": ["Djellaba Moderne", "T-shirt Oversize", "Jean Slim Fit", "Basket Nike", "Abaya Soie", "Caftan de fête", "Chemise en lin", "Sweat à capuche", "Sandales Cuir", "Robe d'été", "Veste Denim", "Costume Homme", "Pyjama Satin", "Echarpe Cachemire", "Ceinture Cuir"],
    "Furniture": ["Tapis Berbère", "Lampe de chevet", "Table basse bois", "Canapé 3 places", "Etagère Murale", "Miroir Doré", "Chaise de bureau", "Bureau d'angle", "Rideaux Occultants", "Cadre déco", "Lit Double", "Armoire 2 portes", "Pouf Salon", "Console Entrée", "Vase Céramique"],
    "Accessories": ["Montre Luxe", "Sac à main cuir", "Lunettes de soleil", "Portefeuille", "Bijoux Argent", "Casquette", "Parapluie", "Valise Voyage", "Coque iPhone", "Porte-clés"]
}

# --- LOGISTIQUE (Scrapée) & GÉOGRAPHIE ---
shipping_data = {
    "Casablanca": {"rate": 35, "hcp": 95}, "Mohammedia": {"rate": 35, "hcp": 85},
    "Rabat": {"rate": 40, "hcp": 90}, "Salé": {"rate": 40, "hcp": 75}, "Kénitra": {"rate": 45, "hcp": 65},
    "Marrakech": {"rate": 45, "hcp": 80}, "Fès": {"rate": 50, "hcp": 60}, "Meknès": {"rate": 50, "hcp": 60},
    "Tanger": {"rate": 50, "hcp": 75}, "Agadir": {"rate": 55, "hcp": 70}, "Oujda": {"rate": 60, "hcp": 50},
    "Laâyoune": {"rate": 70, "hcp": 55}
}

logistique_scraped = {
    "Speedaf Express": 2.1, "Aramex": 2.5, "CTM Messagerie": 4.2, 
    "Livreur Privé": 4.8, "Ghazala": 3.9, "Colis Privé": 4.1
}

marketing_metrics = {
    "Meta (FB/IG)": {"cac": 35, "ctr": 0.025}, "TikTok Ads": {"cac": 22, "ctr": 0.045},
    "Google Search": {"cac": 45, "ctr": 0.015}, "SEO/Direct": {"cac": 5, "ctr": 0.005}
}

motifs_retour = ["Client injoignable", "Refus à la livraison", "Délai trop long", "Produit non conforme", "Changement d'avis"]

# ==========================================
# 2. GÉNÉRATION DE LA DATA
# ==========================================
data = []
start_date = datetime(2023, 1, 1)

for i in range(n_rows):
    # Identité & Genre (Fidélité)
    gender = "Femme" if random.random() > 0.48 else "Homme"
    name = random.choice(femmes) if gender == "Femme" else random.choice(hommes)
    age = random.randint(20, 68)
    type_client = "Fidèle" if random.random() > 0.7 else "Nouveau"
    
    # Géographie & HCP
    city = random.choice(list(shipping_data.keys()))
    ship_cost = shipping_data[city]["rate"]
    hcp_index = shipping_data[city]["hcp"]
    
    # Date & Saisonnalité Maroc
    date = start_date + timedelta(days=random.randint(0, 450))
    month = date.month
    
    # Produit & Prix
    cat = random.choice(list(produits.keys()))
    prod = random.choice(produits[cat])
    qty = random.randint(1, 3)
    
    # Prix de vente (Basé sur tes 1500 lignes)
    if cat == "Electronics": sales = random.randint(500, 9000)
    elif cat == "Furniture": 
        sales = random.randint(300, 5000)
        ship_cost += 40 # Surcharge mobilier
    else: sales = random.randint(50, 1200)
    
    cost_price = sales * random.uniform(0.55, 0.65) # Marge brute ~40%
    
    # Marketing Funnel
    channel = random.choice(list(marketing_metrics.keys()))
    cac = marketing_metrics[channel]["cac"] * random.uniform(0.9, 1.1)
    impressions = random.randint(100, 1000)
    clics = int(impressions * marketing_metrics[channel]["ctr"] * random.uniform(0.5, 1.5))
    
    # Logistique (Basé sur scraping Google Maps)
    partner = random.choice(list(logistique_scraped.keys()))
    score_google = logistique_scraped[partner]
    
    # Succès de livraison
    pay_mode = "Paiement à la livraison" if random.random() > 0.2 else "Carte Bancaire"
    success_rate = 0.5 + (score_google / 10)
    if pay_mode == "Carte Bancaire": success_rate += 0.15 # Moins de retours si déjà payé
    
    rand_status = random.random()
    if rand_status < success_rate:
        status = "Livré"
        motif = "N/A"
        net_profit = sales - cost_price - ship_cost - cac
    elif rand_status < success_rate + 0.10:
        status = "Annulé"
        motif = "Changement d'avis"
        net_profit = -cac
    else:
        status = "Retourné"
        motif = random.choice(motifs_retour)
        if score_google < 3: motif = "Délai trop long"
        net_profit = -(ship_cost + cac) # Perte transport + marketing

    data.append([
        f"M-{10000+i}", date.strftime("%Y-%m-%d"), name, gender, age, type_client,
        city, hcp_index, cat, prod, qty, round(sales, 2), round(cost_price, 2),
        pay_mode, channel, round(cac, 2), impressions, clics,
        partner, score_google, status, motif, round(net_profit, 2)
    ])

# ==========================================
# 3. EXPORTATION
# ==========================================
cols = ["ID_Commande", "Date", "Client", "Genre", "Age", "Type_Client", 
        "Ville", "Indice_HCP", "Categorie", "Produit", "Quantite", "Ventes_Brutes", "Cout_Produit",
        "Mode_Paiement", "Canal_Marketing", "CAC", "Impressions", "Clics",
        "Transporteur", "Note_Google_Maps", "Statut_Livraison", "Motif_Retour", "Profit_Net"]

df_final = pd.DataFrame(data, columns=cols)
df_final.to_csv("PFE_Data_Marketing_Maroc_80K.csv", index=False, encoding='utf-8-sig')

print("Fichier final généré : 80 000 lignes prêtes pour l'analyse expert.")

Fichier final généré : 80 000 lignes prêtes pour l'analyse expert.


In [4]:
import pandas as pd

# 1. Chargement du fichier initial (le CSV extrait)
file_path = 'data fil rouge.csv'
df = pd.read_csv(file_path)

# 2. Conversion simple au format Date propre (puisqu'elles sont déjà textuelles)
df['Date_Convertie'] = pd.to_datetime(df['Date']).dt.date

print("--- Début de la séparation des tables ---")

# 3. CRÉATION DE LA DIMENSION CLIENTS
dim_clients = df[['Client', 'Genre', 'Age', 'Type_Client', 'Ville', 'Indice_HCP']].drop_duplicates().reset_index(drop=True)
dim_clients.index.name = 'ID_Client'
dim_clients = dim_clients.reset_index()

# Liaison de l'ID_Client dans le DataFrame principal
df_liaison = df.merge(dim_clients, on=['Client', 'Genre', 'Age', 'Type_Client', 'Ville', 'Indice_HCP'], how='left')

# 4. CRÉATION DE LA DIMENSION PRODUITS
dim_produits = df[['Produit', 'Categorie']].drop_duplicates().reset_index(drop=True)
dim_produits.index.name = 'ID_Produit'
dim_produits = dim_produits.reset_index()

# Liaison de l'ID_Produit
df_liaison = df_liaison.merge(dim_produits, on=['Produit', 'Categorie'], how='left')

# 5. CRÉATION DE LA DIMENSION TRANSPORTEURS (Scraping Google Maps)
dim_transporteurs = df[['Transporteur', 'Note_Google_Maps']].drop_duplicates().reset_index(drop=True)
dim_transporteurs.index.name = 'ID_Transporteur'
dim_transporteurs = dim_transporteurs.reset_index()

# Liaison de l'ID_Transporteur
df_liaison = df_liaison.merge(dim_transporteurs, on=['Transporteur', 'Note_Google_Maps'], how='left')

# 6. CRÉATION DE LA TABLE DE FAITS MARKETING
fait_marketing = df_liaison[['ID_Commande', 'Canal_Marketing', 'CAC', 'Impressions', 'Clics']].copy()

# 7. CRÉATION DE LA TABLE DE FAITS VENTES ÉPURÉE
fait_ventes_epure = df_liaison[[
    'ID_Commande', 'Date_Convertie', 'ID_Client', 'ID_Produit', 'ID_Transporteur',
    'Quantite', 'Ventes_Brutes', 'Cout_Produit', 'Mode_Paiement',
    'Statut_Livraison', 'Motif_Retour', 'Profit_Net'
]].rename(columns={'Date_Convertie': 'Date_Commande'})

# 8. EXPORTATION EN FICHIERS CSV (avec encodage pour les accents marocains)
dim_clients.to_csv('dim_clients.csv', index=False, encoding='utf-8-sig')
dim_produits.to_csv('dim_produits.csv', index=False, encoding='utf-8-sig')
dim_transporteurs.to_csv('dim_transporteurs.csv', index=False, encoding='utf-8-sig')
fait_marketing.to_csv('fait_marketing.csv', index=False, encoding='utf-8-sig')
fait_ventes_epure.to_csv('fait_ventes_epure.csv', index=False, encoding='utf-8-sig')

print("🎉 Succès ! Les 5 fichiers suivants ont été créés dans votre dossier :")
print("- dim_clients.csv")
print("- dim_produits.csv")
print("- dim_transporteurs.csv")
print("- fait_marketing.csv")
print("- fait_ventes_epure.csv")

--- Début de la séparation des tables ---
🎉 Succès ! Les 5 fichiers suivants ont été créés dans votre dossier :
- dim_clients.csv
- dim_produits.csv
- dim_transporteurs.csv
- fait_marketing.csv
- fait_ventes_epure.csv
